# Let's work on finding the correlation between measles coverage and incidence

### Let's use
- rada_notebooks/nis_measles_vacc_coverage.csv (Primary coverage)
- raw/Vaccination_Coverage_and_Exemptions_among_Kindergartners_20260827.csv (Secondary coverage)
- app/data/tycho_measles_control.csv, Tycho Measles (Primary incidents)

### Start with measles incidence/cases - we are now using Tycho's API for the most recent data

In [1]:
import pandas as pd

tycho_incidents_df = pd.read_csv('../app/data/tycho_measles_control.csv')
tycho_incidents_df.head()

,week,state,cases,year
0,1,NY,376.0,1931
1,1,OR,67.0,1931
2,1,CO,41.0,1931
3,1,AZ,50.0,1931
4,1,MO,1160.0,1931


### Let's start with graphing average measles incidience: year x cases

In [2]:
year_cases_tycho_df = tycho_incidents_df.groupby(by='year')['cases'].mean().reset_index()
year_cases_tycho_df.head()

,year,cases
0,1931,184.207077
1,1932,161.751344
2,1933,156.138298
3,1934,298.681465
4,1935,295.674304


In [3]:
year_cases_tycho_df['year'].min(), year_cases_tycho_df['year'].max()

(np.int64(1931), np.int64(1992))

In [4]:
import plotly.express as px

year_cases_tycho_df_fig = px.line(
    year_cases_tycho_df,
    x="year",
    y="cases",
    markers=True,
    title="Avg. Cases per Year"
)

year_cases_tycho_df_fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Avg. Measles Cases",
)

year_cases_tycho_df_fig.show()

## Let's focus on our latest measles cases from Project Tycho

In [5]:
measles_df = pd.read_csv('../app/data/latest_measles_cases.csv')
measles_df.head()

,cases,year,state
0,85,1927,WI
1,120,1927,WI
2,84,1927,WI
3,106,1927,WI
4,39,1927,WI


In [6]:
year_measles_df = measles_df.groupby(by='year')['cases'].mean().reset_index()
year_measles_df.head()

,year,cases
0,1888,1.857143
1,1889,2.941176
2,1890,2.430976
3,1891,2.614334
4,1892,2.225410


In [7]:
year_measles_df['year'].min(), year_measles_df['year'].max()

(np.int64(1888), np.int64(2001))

In [8]:
year_measles_df_fig = px.line(
    year_measles_df,
    x="year",
    y="cases",
    markers=True,
    title="Avg. Cases per Year"
)

year_measles_df_fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Avg. Measles Cases",
)

year_measles_df_fig.show()

### Our primary coverage will be NIS Child (1995 - 2024)

In [9]:
nis_coverage_df = pd.read_csv('../app/data/nis_measles_vacc_coverage.csv')
nis_coverage_df.head()

,state,year,n,n_vaccinated,coverage_pct
0,AK,1995,194.0,177.0,89.86
1,AK,1996,260.0,225.0,84.55
2,AK,1997,291.0,257.0,87.41
3,AK,1998,34.0,30.0,87.06
4,AK,1999,349.0,321.0,90.67


In [10]:
nis_coverage_df['year'].min(), nis_coverage_df['year'].max()

(np.int64(1995), np.int64(2024))

In [11]:
year_coverage_nis_df = nis_coverage_df.groupby(by='year')['coverage_pct'].mean().reset_index()
year_coverage_nis_df.head()

,year,coverage_pct
0,1995,89.875882
1,1996,90.327647
2,1997,90.728431
3,1998,92.482549
4,1999,91.712353


In [12]:
nis_coverage_df_fig = px.line(
    year_coverage_nis_df,
    x="year",
    y="coverage_pct",
    markers=True,
    title="Avg. Vaccination Coverage per Year"
)

nis_coverage_df_fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Avg. Vaccination Coverage",
)

nis_coverage_df_fig.show()

In [13]:
year_measles_df_fig.show()

## Recall what these columns mean:

| Column         | Level          | What it means                              | Why you need it                                                                     |
| -------------- | -------------- | ------------------------------------------ | ----------------------------------------------------------------------------------- |
| **`SEQNUMC`**  | Child          | Unique child identifier                    | Identifies the individual child/record                                              |
| **`SEQNUMHH`** | Household      | Household identifier                       | Identifies which household the child belongs to; used in survey design              |
| **`STRATUM`**  | Sampling group | Survey sampling stratum                    | Identifies the sampling group; needed for correct SEs/CIs                           |
| **`PROVWT`**   | Child          | Provider-phase survey weight               | Determines how much the child contributes to population estimates                   |
| **`P_UTDMCV`** | Child          | Measles vaccination status                 | `1` = received ≥1 qualifying measles-containing vaccination; `0` = did not          |
| **`P_NUMMMR`** | Child          | Number of measles-containing vaccine doses | Shows how many provider-reported measles-containing vaccinations the child received |
| **`STATE`**    | Geography      | State code                                 | Lets you calculate vaccination coverage by state                                    |
| **`YEAR`**     | Time           | Survey year                                | Lets you calculate and compare coverage over time                                   |

# Let's now merge these by state and year
- measles_df
- year_coverage_nis_df

In [27]:
annual_measles_df = (
    measles_df
    .groupby(["state", "year"], as_index=False)
    .agg(cases=("cases", "sum"))
)

In [50]:
cases_and_coverage_df = pd.merge(
    annual_measles_df,
    nis_coverage_df,
    on=["state", "year"],
    how="left"
)

cases_and_coverage_df.head()

,state,year,cases,n,n_vaccinated,coverage_pct
0,AK,1914,1,NaN,NaN,NaN
1,AK,1954,1487,NaN,NaN,NaN
2,AK,1955,537,NaN,NaN,NaN
3,AK,1956,2511,NaN,NaN,NaN
4,AK,1957,958,NaN,NaN,NaN


In [ ]:
cases_and_coverage_df.duplicated(
    subset=["state", "year"]
).sum()

np.int64(0)

In [52]:
len(cases_and_coverage_df)

4593

In [53]:
year_summary_df = (
    cases_and_coverage_df
    .groupby("year", as_index=False)
    .agg(
        cases=("cases", "sum"),
        n=("n", "sum"),
        n_vaccinated=("n_vaccinated", "sum")
    )
)

year_summary_df["coverage_pct"] = (
    year_summary_df["n_vaccinated"]
    / year_summary_df["n"]
    * 100
)

In [54]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=year_summary_df["year"],
        y=year_summary_df["coverage_pct"],
        name="Vaccination coverage",
        mode="lines+markers"
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=year_summary_df["year"],
        y=year_summary_df["cases"],
        name="Measles cases",
        mode="lines+markers"
    ),
    secondary_y=True
)

fig.update_layout(
    title="Vaccination Coverage and Measles Cases per Year",
    xaxis_title="Year",
    hovermode="x unified"
)

fig.update_yaxes(
    title_text="Vaccination Coverage (%)",
    secondary_y=False
)

fig.update_yaxes(
    title_text="Total Measles Cases",
    secondary_y=True
)

fig.show()

# As you can see here vacc coverage should at least be covering from 1964. Here is what ChatGPT recommends to get more data:
| Period       | Source                                                                                                                                                                                               | Likely geographic resolution                                |
| ------------ | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------- |
| 1964–1985    | [United States Immunization Survey (historical CDC summary)](https://stacks.cdc.gov/view/cdc/64850/cdc_64850_DS1.pdf)                                                                                | Primarily national/regional published estimates             |
| 1986–1990    | Surveillance gap                                                                                                                                                                                     | Limited estimates scattered across CDC publications         |
| 1991–1994    | [NHIS via IPUMS](https://nhis.ipums.org/) · [NHIS via ICPSR](https://www.icpsr.umich.edu/web/NACDA/series/40)                                                                                        | Primarily national; microdata may permit regional estimates |
| 1995–2024    | [CDC NIS-Child data](https://www.cdc.gov/nis/data-tables/index.html)                                                                                                                                 | State and national; children aged 19–35 months              |
| 2009–present | [CDC SchoolVaxView data](https://www.cdc.gov/schoolvaxview/data/index.html) · [Downloadable dataset](https://data.cdc.gov/Vaccinations/Vaccination-Coverage-and-Exemptions-among-Kinderga/ijqb-a7ye) | State and national; kindergartners                          |

**We will worry about 1986-1990 later on...**

In [55]:
assert False

AssertionError: 

# Let's start with SchoolVaxView to append to our NIS Child data

In [57]:
import pandas as pd

schoolvaxview_df = pd.read_csv('../raw/school_vax_view/schoolvaxview.csv')
schoolvaxview_df.head()

,vaccine,dose,geography_type,geography,year_season,coverage_estimate,population_sample_size,percent_surveyed,foot_notes,number_of_exemptions,survey_type
0,MMR,NaN,States,Kansas,2022-23,91.6,35543.0,30.8,†. ‡. §,NaN,Stratified 1-stage cluster sample
1,MMR,NaN,States,Kentucky,2022-23,90.1,54742.0,96.9,>=. ‡. §,NaN,Census
2,MMR,NaN,States,Louisiana,2022-23,92.2,54314.0,100.0,*,NaN,Census
3,MMR,NaN,States,Maine,2022-23,96.8,12403.0,93.9,NaN,NaN,Census
4,MMR,NaN,States,Maryland,2022-23,96.7,59684.0,100.0,*. ‡,NaN,Census


In [58]:
schoolvaxview_df["year"] = (
    schoolvaxview_df["year_season"]
    .str.split("-")
    .str[0]
    .astype(int)
    .add(1)
)

In [59]:
import us


schoolvaxview_df["state"] = schoolvaxview_df["geography"].map(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)


In [60]:
core_columns = [
    'vaccine',
    'dose',
    'geography_type',
    'coverage_estimate',
    'year',
    'state',
]

schoolvaxview_df = schoolvaxview_df[core_columns]
schoolvaxview_df.head()

,vaccine,dose,geography_type,coverage_estimate,year,state
0,MMR,NaN,States,91.6,2023,KS
1,MMR,NaN,States,90.1,2023,KY
2,MMR,NaN,States,92.2,2023,LA
3,MMR,NaN,States,96.8,2023,ME
4,MMR,NaN,States,96.7,2023,MD


In [61]:
measles_schoolvaxview_df = schoolvaxview_df[(schoolvaxview_df['vaccine'] == 'MMR')
                                            & (schoolvaxview_df['geography_type'] == 'States')]
len(measles_schoolvaxview_df)

868

In [62]:
measles_schoolvaxview_df.head()

,vaccine,dose,geography_type,coverage_estimate,year,state
0,MMR,NaN,States,91.6,2023,KS
1,MMR,NaN,States,90.1,2023,KY
2,MMR,NaN,States,92.2,2023,LA
3,MMR,NaN,States,96.8,2023,ME
4,MMR,NaN,States,96.7,2023,MD


In [63]:
measles_schoolvaxview_df['year'].min(), measles_schoolvaxview_df['year'].max()

(np.int64(2010), np.int64(2026))

In [64]:
measles_schoolvaxview_df = measles_schoolvaxview_df[['coverage_estimate', 'year', 'state']]
measles_schoolvaxview_df.head()

,coverage_estimate,year,state
0,91.6,2023,KS
1,90.1,2023,KY
2,92.2,2023,LA
3,96.8,2023,ME
4,96.7,2023,MD


In [77]:
measles_schoolvaxview_df.duplicated(
    subset=["state", "year"]
).sum()

np.int64(0)

In [76]:
measles_schoolvaxview_df = (
    measles_schoolvaxview_df
    .drop_duplicates(
        subset=["state", "year"],
        keep="first"
    )
    .reset_index(drop=True)
)

In [78]:
measles_schoolvaxview_df.to_csv('../app/data/measles_schoolvaxview.csv', index=False)

## Let's work on 1991 to 1994
| Year | Coverage | Definition                                                                     | Source                                                                                         |
| ---: | -------: | ------------------------------------------------------------------------------ | ---------------------------------------------------------------------------------------------- |
| 1991 |    82.0% | ≥1 measles-containing dose, ages 19–35 months                                  | [CDC NHIS report](https://www.cdc.gov/mmwr/preview/mmwrhtml/00024988.htm)                      |
| 1992 |    82.5% | ≥1 measles-containing dose, ages 19–35 months                                  | [CDC NHIS report](https://www.cdc.gov/mmwr/preview/mmwrhtml/00024988.htm)                      |
| 1993 |    80.8% | ≥1 measles-containing dose, ages 19–35 months; provisional first-half estimate | [CDC NHIS report](https://www.cdc.gov/mmwr/preview/mmwrhtml/00026290.htm)                      |
| 1994 |    91.0% | ≥1 measles-containing dose, ages 19–35 months; NHIS January–June               | [CDC comparison table](https://restoredcdc.org/www.cdc.gov/mmwr/preview/mmwrhtml/00038532.htm) |
| 1994 |    89.0% | ≥1 measles-containing dose, ages 19–35 months; NIS April–December              | [CDC comparison table](https://restoredcdc.org/www.cdc.gov/mmwr/preview/mmwrhtml/00038532.htm) |

***This data was parsed by ChatGPT from these reports***


In [82]:
historical_nhis_df = pd.DataFrame({
    "year": [1991, 1992, 1993, 1994],
    "coverage_pct": [82.0, 82.5, 80.8, 89.0],
    "source": ["NHIS", "NHIS", "NHIS", "NIS"],
    "coverage_definition": [
        "1+ measles-containing vaccine",
        "1+ measles-containing vaccine",
        "1+ measles-containing vaccine",
        "1+ measles-containing vaccine"
    ],
    "geography": ["US", "US", "US", "US"]
})

historical_nhis_df = historical_nhis_df[['year', 'coverage_pct']]
historical_nhis_df

,year,coverage_pct
0,1991,82.0
1,1992,82.5
2,1993,80.8
3,1994,89.0


In [83]:
historical_nhis_df.to_csv('../app/data/historical_nhis.csv', index=False)

# Now let's work on years 1986-1990
Confirmed by ChatGPT as genuine missing gap

Update plotly later...
```
go.Scatter(
    x=coverage_df["year"],
    y=coverage_df["coverage_pct"],
    connectgaps=False
)
```

In [86]:
coverage_gap_df = pd.DataFrame({
    "year": range(1986, 1991),
    "coverage_pct": [pd.NA] * 5,
    "source": ["No national surveillance data"] * 5,
    "coverage_type": ["missing"] * 5,
    "geography": ["US"] * 5
})
coverage_gap_df = coverage_gap_df[['year', 'coverage_pct']]
coverage_gap_df

,year,coverage_pct
0,1986,<NA>
1,1987,<NA>
2,1988,<NA>
3,1989,<NA>
4,1990,<NA>


In [87]:
coverage_gap_df.to_csv('../app/data/coverage_gap.csv', index=False)

# Last, 1967 to 1985


In [88]:
import pandas as pd

missing_coverage_df = pd.DataFrame({
    "year": list(range(1964, 1967)) + list(range(1986, 1991)),
    "coverage_pct": pd.NA,
    "source": (
        ["USIS value unavailable"] * 3
        + ["No national surveillance data"] * 5
    ),
    "coverage_type": ["missing"] * 8,
    "geography": ["US"] * 8
})

missing_coverage_df

,year,coverage_pct,source,coverage_type,geography
0,1964,<NA>,USIS value unavailable,missing,US
1,1965,<NA>,USIS value unavailable,missing,US
2,1966,<NA>,USIS value unavailable,missing,US
3,1986,<NA>,No national surveillance data,missing,US
4,1987,<NA>,No national surveillance data,missing,US
5,1988,<NA>,No national surveillance data,missing,US
6,1989,<NA>,No national surveillance data,missing,US
7,1990,<NA>,No national surveillance data,missing,US


In [91]:
import pandas as pd

usis_coverage_df = pd.DataFrame({
    "year": range(1967, 1986),
    "coverage_pct": pd.NA,
    "source": "United States Immunization Survey",
    "coverage_type": "unavailable",
    "geography": "US"
})

verified_usis_values = {
    1982: 67.0,
    1985: 61.0
}

usis_coverage_df["coverage_pct"] = (
    usis_coverage_df["year"].map(verified_usis_values)
)

usis_coverage_df.loc[
    usis_coverage_df["coverage_pct"].notna(),
    "coverage_type"
] = "published_anchor"

usis_coverage_df

,year,coverage_pct,source,coverage_type,geography
0,1967,NaN,United States Immunization Survey,unavailable,US
1,1968,NaN,United States Immunization Survey,unavailable,US
2,1969,NaN,United States Immunization Survey,unavailable,US
3,1970,NaN,United States Immunization Survey,unavailable,US
4,1971,NaN,United States Immunization Survey,unavailable,US
5,1972,NaN,United States Immunization Survey,unavailable,US
6,1973,NaN,United States Immunization Survey,unavailable,US
7,1974,NaN,United States Immunization Survey,unavailable,US
8,1975,NaN,United States Immunization Survey,unavailable,US
9,1976,NaN,United States Immunization Survey,unavailable,US
